# Notebook for Library merging

In [1]:
# Imports
from itertools import product
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config
from Models.dataset_and_models import Dataset, Model

In [5]:
# Load configurations
CONFIG = "../configs/config.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}

In [6]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [7]:
# Full cross-product: one entry per benchmark run
benchmark_runs = list(product(model_runs, dataset_runs))
print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {len(benchmark_runs)} total runs")

# Load a single dataset variant
dataset_key, dataset_params = dataset_runs[0]
ds = Dataset[dataset_key.upper()].load_dataset(**dataset_params)
X, y = ds["X"], ds["y"]

4 model configs × 324 dataset configs = 1296 total runs


### Iterate trough every combination and train the model for explanation

In [ ]:
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import (
    ShapTrueValueBackend,
    ShapApproxBackend,
    ShapIQApproxBackend,
    LightShapApproxBackend,
)

# Exact ground truth: shap.Explainer auto-detects TreeSHAP / LinearSHAP — polynomial,
# exact, and (with background data) interventional == the marginal value function that
# every approximator below targets. One cheap oracle per cell, recomputed each run.
TRUE_VALUE_BACKENDS = [ShapTrueValueBackend]

# Approximation sweep: one config per (library × approximator × budget).
# The nominal budget differs in units across libraries, so n_model_evals — the real
# count of model calls, recorded by the runner — is the comparable budget axis.
BUDGETS = [256, 512, 1024, 2048, 4096]
APPROX_BACKENDS = [ShapApproxBackend, ShapIQApproxBackend, LightShapApproxBackend]

approximation_specs = [
    (backend, {"approximator": approximator, "budget": budget})
    for backend in APPROX_BACKENDS
    for approximator in ("kernel", "permutation")
    for budget in BUDGETS
]

runner = BenchmarkRunner(
    true_value_backends=TRUE_VALUE_BACKENDS,
    approximation_specs=approximation_specs,
    output_csv="../Benchmarking/results.csv",
    n_background=100,
    n_eval=None,
)

for dataset_key, dataset_params in tqdm(dataset_runs, desc="datasets"):
    dataset_enum = Dataset[dataset_key.upper()]
    ds = dataset_enum.load_dataset(**dataset_params)
    X, y = ds["X"], ds["y"]

    for model_key, model_params in tqdm(model_runs, leave=False, desc="models"):
        trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params)
        trainer.fit(X, y, task=ds["task"])

        runner.run(
            model=trainer.get_model(),
            X=X,
            run_meta={
                "dataset": dataset_key,
                "model": model_key,
                "n_features": dataset_params.get("n_features"),
                "n_samples": dataset_params.get("n_samples"),
            },
        )

print("Done. Results written to ../Benchmarking/results.csv")

In [ ]:
import pandas as pd

results = pd.read_csv("../Benchmarking/results.csv")
print(f"{len(results)} rows written")

cols = [
    "dataset", "model", "n_features", "backend", "approximator", "budget",
    "n_model_evals", "runtime_s", "mean_abs_diff", "relative_mae",
    "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)

## Ground-truth validation (run once)

The exact reference used across the whole sweep is `shap`'s auto-detected **polynomial**
explainer (TreeSHAP / LinearSHAP) — fast at any feature count. To confirm this cheap oracle
really equals the exact Shapley value, compare it on a small problem (california-housing,
8 features) against an **independent brute-force exact**: `shapiq`'s general model-agnostic
explainer at the maximum budget `2⁸ = 256` (full coalition enumeration) — a different
library *and* a different algorithm. Agreement to numerical precision validates the oracle.
One-off check, **not** part of the main sweep.

In [ ]:
from Benchmarking.backends import ShapTrueValueBackend, ShapIQTrueValueBackend

val_ds = Dataset.CALIFORNIA_HOUSING.load_dataset(n_samples=300, n_features=8)
Xv, yv = val_ds["X"], val_ds["y"]
val_trainer = Model.RANDOM_FOREST.get_model_with_params(
    Dataset.CALIFORNIA_HOUSING, {"n_estimators": 50, "max_depth": 6}
)
val_trainer.fit(Xv, yv, task=val_ds["task"])
val_model = val_trainer.get_model()

bg, xe = Xv.iloc[:100], Xv.iloc[100:120]

# Oracle: polynomial TreeSHAP (the reference used across the whole sweep)
poly = ShapTrueValueBackend(val_model, bg).run_explainer(xe)

# Independent brute-force exact: shapiq's general model-agnostic explainer at the maximum
# budget = 2^n_features (full coalition enumeration) — different library, different algorithm.
shapiq_exact = ShapIQTrueValueBackend(val_model, bg).run_explainer(xe)

print("max |TreeSHAP - shapiq exact (full budget)|:", float((poly - shapiq_exact).abs().values.max()))
print("-> agree to numerical precision; oracle validated.")